# MobileNetV1 `.pth` → ONNX → TensorFlow SavedModel → TensorFlow Lite

此 Notebook 專門對應你的自訂 `MobileNetV1` 架構：

- 輸入：`(1, 3, 224, 224)`
- 輸出：`(1, 100)`
- 權重檔：`C:\Users\user\Desktop\pth_to_TFLite\MobileNet-V1_imagenet100.pth`
- 輸出資料夾：`C:\Users\user\Desktop\pth_to_TFLite\MobileNetV1_pth_to_TFLite`

> 建議環境：你的 `torch310` conda 環境。

In [1]:
# =========================================================
# 0. 可選：安裝/補齊必要套件
# =========================================================
# 第一次執行若缺套件，請取消下一行註解後執行。
# 注意：安裝完若仍 import 失敗，請 Restart Kernel 後再從頭執行。
# !pip install onnx onnxscript onnxruntime onnx-tf tensorflow tensorflow-probability

print("如缺套件，請取消上面 pip install 那行的註解後執行。")

如缺套件，請取消上面 pip install 那行的註解後執行。


In [ ]:
# =========================================================
# 1. 路徑與基本設定
# =========================================================
from pathlib import Path
import os
import json
import shutil
import subprocess
import sys
import importlib.util

BASE_INPUT_DIR = Path(r"C:\Users\user\Desktop\pth_to_TFLite")
PTH_PATH = BASE_INPUT_DIR / "MobileNet-V1_imagenet100.pth"

OUT_DIR = BASE_INPUT_DIR / "MobileNet-V1_pth_to_TFLite"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ONNX_PATH = OUT_DIR / "model.onnx"
TF_DIR = OUT_DIR / "model_tf"
TFLITE_PATH = OUT_DIR / "MobileNet-V1_imagenet100.tflite"
INFO_PATH = OUT_DIR / "tflite_model_info.json"

print("PTH_PATH :", PTH_PATH)
print("OUT_DIR  :", OUT_DIR)
print("ONNX_PATH:", ONNX_PATH)
print("TF_DIR   :", TF_DIR)
print("TFLITE   :", TFLITE_PATH)

if not PTH_PATH.exists():
    raise FileNotFoundError(f"找不到 pth 檔案：{PTH_PATH}")

print("✅ pth 檔案存在")

PTH_PATH : C:\Users\user\Desktop\pth_to_TFLite\MobileNet-V1_imagenet100.pth
OUT_DIR  : C:\Users\user\Desktop\pth_to_TFLite\MobileNetV1_pth_to_TFLite
ONNX_PATH: C:\Users\user\Desktop\pth_to_TFLite\MobileNetV1_pth_to_TFLite\model.onnx
TF_DIR   : C:\Users\user\Desktop\pth_to_TFLite\MobileNetV1_pth_to_TFLite\model_tf
TFLITE   : C:\Users\user\Desktop\pth_to_TFLite\MobileNetV1_pth_to_TFLite\MobileNet-V1_imagenet100.tflite
✅ pth 檔案存在


In [3]:
# =========================================================
# 2. 檢查套件版本
# =========================================================
def check_module(name):
    spec = importlib.util.find_spec(name)
    return spec is not None

required = ["torch", "onnx", "tensorflow"]
optional = ["onnxruntime", "onnx_tf", "tensorflow_probability", "onnxscript"]

for m in required + optional:
    print(f"{m:24s}:", "OK" if check_module(m) else "MISSING")

import torch
import torch.nn as nn
print("\ntorch:", torch.__version__)

try:
    import onnx
    print("onnx :", onnx.__version__)
except Exception as e:
    print("onnx import error:", repr(e))

try:
    import tensorflow as tf
    print("tensorflow:", tf.__version__)
except Exception as e:
    print("tensorflow import error:", repr(e))

torch                   : OK
onnx                    : OK
tensorflow              : OK
onnxruntime             : OK
onnx_tf                 : OK
tensorflow_probability  : OK
onnxscript              : MISSING

torch: 2.11.0+cu128
onnx : 1.13.1
tensorflow: 2.13.1


In [4]:
# =========================================================
# 3. 定義 MobileNetV1 架構（對齊你的訓練/轉 pt 程式）
# =========================================================
import torch
import torch.nn as nn

class MobileNetV1(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()

        def conv_bn(inp, oup, stride):
            return nn.Sequential(
                nn.Conv2d(inp, oup, 3, stride, 1, bias=False),
                nn.BatchNorm2d(oup),
                nn.ReLU6(inplace=True)
            )

        def conv_dw(inp, oup, stride):
            return nn.Sequential(
                nn.Conv2d(inp, inp, 3, stride, 1, groups=inp, bias=False),
                nn.BatchNorm2d(inp),
                nn.ReLU6(inplace=True),
                nn.Conv2d(inp, oup, 1, 1, 0, bias=False),
                nn.BatchNorm2d(oup),
                nn.ReLU6(inplace=True)
            )

        self.model = nn.Sequential(
            conv_bn(3, 32, 2),
            conv_dw(32, 64, 1),
            conv_dw(64, 128, 2),
            conv_dw(128, 128, 1),
            conv_dw(128, 256, 2),
            conv_dw(256, 256, 1),
            conv_dw(256, 512, 2),
            *[conv_dw(512, 512, 1) for _ in range(5)],
            conv_dw(512, 1024, 2),
            conv_dw(1024, 1024, 1),
            nn.AdaptiveAvgPool2d(1)
        )
        self.dropout = nn.Dropout(p=0.2)
        self.fc = nn.Linear(1024, num_classes)

    def forward(self, x):
        x = self.model(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        return self.fc(x)

print("✅ MobileNetV1 class ready")

✅ MobileNetV1 class ready


In [5]:
# =========================================================
# 4. 載入 .pth 權重
# =========================================================
device = torch.device("cpu")
num_classes = 100

model = MobileNetV1(num_classes=num_classes).to(device)

checkpoint = torch.load(PTH_PATH, map_location=device)

# 支援 checkpoint / state_dict / model_state_dict
if isinstance(checkpoint, dict):
    if "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    elif "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    else:
        state_dict = checkpoint
else:
    state_dict = checkpoint

# 去除 DataParallel 的 module. 前綴
new_state_dict = {}
for k, v in state_dict.items():
    if k.startswith("module."):
        k = k.replace("module.", "", 1)
    new_state_dict[k] = v

missing, unexpected = model.load_state_dict(new_state_dict, strict=False)

print("missing keys   :", missing)
print("unexpected keys:", unexpected)

if missing or unexpected:
    raise RuntimeError("權重載入不完全，請確認 pth 是否對應 MobileNetV1 架構。")

model.eval()
print("✅ pth 權重載入成功")

missing keys   : []
unexpected keys: []
✅ pth 權重載入成功


In [6]:
# =========================================================
# 5. PyTorch shape 測試
# =========================================================
dummy = torch.randn(1, 3, 224, 224, device=device)

with torch.no_grad():
    y = model(dummy)

print("input shape :", tuple(dummy.shape))
print("output shape:", tuple(y.shape))

if tuple(y.shape) != (1, 100):
    raise RuntimeError(f"輸出 shape 錯誤：{tuple(y.shape)}，預期 (1, 100)")

print("✅ PyTorch forward OK")

input shape : (1, 3, 224, 224)
output shape: (1, 100)
✅ PyTorch forward OK


In [7]:
# =========================================================
# 6. 匯出 ONNX
# 重要：dynamo=False，避免新版 PyTorch 先匯出 opset18 再降版導致 conversion 問題
# =========================================================
if ONNX_PATH.exists():
    ONNX_PATH.unlink()

torch.onnx.export(
    model,
    dummy,
    str(ONNX_PATH),
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes=None,
    dynamo=False
)

print("✅ ONNX export done:", ONNX_PATH)
print("ONNX size MB:", ONNX_PATH.stat().st_size / 1024 / 1024)

C:\Users\user\AppData\Local\Temp\ipykernel_18524\335016881.py:8: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


✅ ONNX export done: C:\Users\user\Desktop\pth_to_TFLite\MobileNetV1_pth_to_TFLite\model.onnx
ONNX size MB: 12.60287094116211


In [8]:
# =========================================================
# 7. ONNX checker + ONNXRuntime 測試
# =========================================================
import numpy as np
import onnx

onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)
print("✅ ONNX checker passed")

try:
    import onnxruntime as ort

    sess = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
    input_name = sess.get_inputs()[0].name
    output_name = sess.get_outputs()[0].name

    x_np = dummy.cpu().numpy().astype(np.float32)
    y_onnx = sess.run([output_name], {input_name: x_np})[0]

    print("ONNX input :", input_name)
    print("ONNX output:", output_name)
    print("ONNX output shape:", y_onnx.shape)

    with torch.no_grad():
        y_pt = model(dummy).cpu().numpy()

    max_abs_diff = float(np.max(np.abs(y_pt - y_onnx)))
    print("PyTorch vs ONNX max_abs_diff:", max_abs_diff)

except Exception as e:
    print("⚠️ ONNXRuntime 測試失敗，但 ONNX 可能仍可轉換：", repr(e))

✅ ONNX checker passed
ONNX input : input
ONNX output: output
ONNX output shape: (1, 100)
PyTorch vs ONNX max_abs_diff: 3.933906555175781e-06


In [9]:
# =========================================================
# 8. ONNX → TensorFlow SavedModel
# =========================================================
# 注意：
# 1) onnx-tf 對 tensorflow / tensorflow-probability 版本敏感。
# 2) 若本 cell 失敗，請完整複製 STDERR 給我。
# =========================================================
if TF_DIR.exists():
    shutil.rmtree(TF_DIR)

cmd = [
    sys.executable, "-m", "onnx_tf.cli",
    "convert",
    "-i", str(ONNX_PATH),
    "-o", str(TF_DIR)
]

print("Running command:")
print(" ".join([f'"{c}"' if " " in c else c for c in cmd]))

result = subprocess.run(cmd, capture_output=True, text=True)

print("STDOUT:\n", result.stdout)
print("STDERR:\n", result.stderr)

if result.returncode != 0:
    raise RuntimeError("ONNX → TensorFlow SavedModel 轉換失敗。請貼上本 cell 的 STDERR。")

if not TF_DIR.exists():
    raise RuntimeError(f"轉換結束但找不到 SavedModel 資料夾：{TF_DIR}")

print("✅ TensorFlow SavedModel done:", TF_DIR)

Running command:
c:\Users\user\anaconda3\envs\torch310\python.exe -m onnx_tf.cli convert -i C:\Users\user\Desktop\pth_to_TFLite\MobileNetV1_pth_to_TFLite\model.onnx -o C:\Users\user\Desktop\pth_to_TFLite\MobileNetV1_pth_to_TFLite\model_tf
STDOUT:
 
STDERR:
 c:\Users\user\anaconda3\envs\torch310\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
2026-04-30 19:44:56,250 - onnx-tf - INFO - Start converting onnx pb to tf saved model
2026-04-30 19:44:56.284238: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use

In [10]:
# =========================================================
# 9. TensorFlow SavedModel → TFLite
# =========================================================
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_saved_model(str(TF_DIR))

# Float32 模型：準確率最穩
# 若想壓縮可保留 Optimize.DEFAULT，但第一次建議先用 Float32 確認正確。
USE_OPTIMIZE_DEFAULT = False

if USE_OPTIMIZE_DEFAULT:
    converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)

print("✅ TFLite done:", TFLITE_PATH)
print("TFLite size MB:", TFLITE_PATH.stat().st_size / 1024 / 1024)

✅ TFLite done: C:\Users\user\Desktop\pth_to_TFLite\MobileNetV1_pth_to_TFLite\MobileNet-V1_imagenet100.tflite
TFLite size MB: 12.603515625


In [11]:
# =========================================================
# 10. TFLite Interpreter 測試
# =========================================================
import numpy as np
import tensorflow as tf

interpreter = tf.lite.Interpreter(model_path=str(TFLITE_PATH))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input details:")
for d in input_details:
    print(d)

print("\nOutput details:")
for d in output_details:
    print(d)

# TFLite 通常使用 NHWC: (1, 224, 224, 3)
# 但如果轉換保留 NCHW，這裡會自動判斷。
input_shape = input_details[0]["shape"]
input_dtype = input_details[0]["dtype"]

x_nchw = dummy.cpu().numpy().astype(np.float32)
x_nhwc = np.transpose(x_nchw, (0, 2, 3, 1))

if list(input_shape) == [1, 3, 224, 224]:
    x_tflite = x_nchw
elif list(input_shape) == [1, 224, 224, 3]:
    x_tflite = x_nhwc
else:
    raise RuntimeError(f"未知 TFLite input shape: {input_shape}")

x_tflite = x_tflite.astype(input_dtype)

interpreter.set_tensor(input_details[0]["index"], x_tflite)
interpreter.invoke()
y_tflite = interpreter.get_tensor(output_details[0]["index"])

print("TFLite output shape:", y_tflite.shape)

if y_tflite.shape[-1] != 100:
    raise RuntimeError(f"TFLite 輸出類別數錯誤：{y_tflite.shape}")

print("✅ TFLite Interpreter 測試 OK")

Input details:
{'name': 'serving_default_input:0', 'index': 0, 'shape': array([  1,   3, 224, 224]), 'shape_signature': array([  1,   3, 224, 224]), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}

Output details:
{'name': 'PartitionedCall:0', 'index': 159, 'shape': array([  1, 100]), 'shape_signature': array([  1, 100]), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}
TFLite output shape: (1, 100)
✅ TFLite Interpreter 測試 OK


In [12]:
# =========================================================
# 11. 儲存模型資訊 JSON
# =========================================================
info = {
    "source_pth": str(PTH_PATH),
    "output_dir": str(OUT_DIR),
    "onnx_path": str(ONNX_PATH),
    "tensorflow_saved_model_dir": str(TF_DIR),
    "tflite_path": str(TFLITE_PATH),
    "input_size": [1, 3, 224, 224],
    "num_classes": 100,
    "preprocess": {
        "resize": 256,
        "center_crop": 224,
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225],
        "note": "Android 端推論必須使用相同 normalize。"
    },
    "tflite_input_details": [
        {
            "name": str(d.get("name")),
            "shape": [int(x) for x in d.get("shape")],
            "dtype": str(d.get("dtype"))
        }
        for d in input_details
    ],
    "tflite_output_details": [
        {
            "name": str(d.get("name")),
            "shape": [int(x) for x in d.get("shape")],
            "dtype": str(d.get("dtype"))
        }
        for d in output_details
    ]
}

with open(INFO_PATH, "w", encoding="utf-8") as f:
    json.dump(info, f, ensure_ascii=False, indent=2)

print("✅ Saved info:", INFO_PATH)
print(json.dumps(info, ensure_ascii=False, indent=2))

✅ Saved info: C:\Users\user\Desktop\pth_to_TFLite\MobileNetV1_pth_to_TFLite\tflite_model_info.json
{
  "source_pth": "C:\\Users\\user\\Desktop\\pth_to_TFLite\\MobileNet-V1_imagenet100.pth",
  "output_dir": "C:\\Users\\user\\Desktop\\pth_to_TFLite\\MobileNetV1_pth_to_TFLite",
  "onnx_path": "C:\\Users\\user\\Desktop\\pth_to_TFLite\\MobileNetV1_pth_to_TFLite\\model.onnx",
  "tensorflow_saved_model_dir": "C:\\Users\\user\\Desktop\\pth_to_TFLite\\MobileNetV1_pth_to_TFLite\\model_tf",
  "tflite_path": "C:\\Users\\user\\Desktop\\pth_to_TFLite\\MobileNetV1_pth_to_TFLite\\MobileNet-V1_imagenet100.tflite",
  "input_size": [
    1,
    3,
    224,
    224
  ],
  "num_classes": 100,
  "preprocess": {
    "resize": 256,
    "center_crop": 224,
    "mean": [
      0.485,
      0.456,
      0.406
    ],
    "std": [
      0.229,
      0.224,
      0.225
    ],
    "note": "Android 端推論必須使用相同 normalize。"
  },
  "tflite_input_details": [
    {
      "name": "serving_default_input:0",
      "shape": [
 

In [13]:
# =========================================================
# 12. 最終檢查輸出檔案
# =========================================================
print("輸出資料夾：", OUT_DIR)
print()

for p in [ONNX_PATH, TF_DIR, TFLITE_PATH, INFO_PATH]:
    if p.exists():
        if p.is_file():
            print("✅", p.name, f"{p.stat().st_size / 1024 / 1024:.3f} MB")
        else:
            print("✅", p.name, "<DIR>")
    else:
        print("❌", p)

輸出資料夾： C:\Users\user\Desktop\pth_to_TFLite\MobileNetV1_pth_to_TFLite

✅ model.onnx 12.603 MB
✅ model_tf <DIR>
✅ MobileNet-V1_imagenet100.tflite 12.604 MB
✅ tflite_model_info.json 0.001 MB
